# §6 MLP regime 1 / Compare Results — regression, equal wall-clock
Loads `results/{backprop,two_factor,three_factor_clean,three_factor_normal,three_factor_noisy}.json` from Drive (run those first; missing ones skipped). Includes the three-factor cos-sweep head-to-head.

## 1. Setup

In [ ]:
import os, json, math, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_DRIVE, DRIVE_SUBDIR = True, 'Section6_regime1'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); STORE = os.path.join('/content/drive/MyDrive', DRIVE_SUBDIR)
    except Exception as e:
        print('Drive mount failed:', e); STORE = os.path.join('/content', DRIVE_SUBDIR)
else:
    STORE = os.path.join('.', DRIVE_SUBDIR)
RESULTS_DIR = os.path.join(STORE, 'results'); CKPT_DIR = os.path.join(STORE, 'checkpoints')
EXTRA_RESULTS_DIRS = []   # merge results JSONs from other Google accounts here (see the "merge" cell)
EXTRA_CKPT_DIRS    = []
METHODS = ['backprop', 'two_factor', 'three_factor_clean', 'three_factor_normal', 'three_factor_noisy', 'backprop_100M']
LABELS = {'backprop':'Backprop', 'two_factor':'Two-factor Hebbian',
          'three_factor_clean':'Three-factor clean (cos~0.5)', 'three_factor_normal':'Three-factor normal (cos~0.09)',
          'three_factor_noisy':'Three-factor noisy (cos~0.01)', 'backprop_100M':'Backprop 100M (ceiling)'}
COLORS = {'backprop':'#1F3864', 'two_factor':'#B8860B', 'three_factor_clean':'#C62828',
          'three_factor_normal':'#E67E22', 'three_factor_noisy':'#7B1FA2', 'backprop_100M':'#2E7D32'}
print('reading', RESULTS_DIR)

## 1b. Merge results from another account (optional)

In [ ]:
# OPTIONAL: merge results exported from ANOTHER Google account (e.g. if backprop.json ran on a different account).
import os
try:
    from google.colab import files
    up = files.upload()                       # pick <method>.json (and optionally <method>.pt)
    import shutil
    for fn in up:
        dst = '/content/extra_results' if fn.endswith('.json') else '/content/extra_ckpts'
        os.makedirs(dst, exist_ok=True); shutil.move(fn, os.path.join(dst, fn))
    if os.path.isdir('/content/extra_results') and '/content/extra_results' not in EXTRA_RESULTS_DIRS: EXTRA_RESULTS_DIRS.append('/content/extra_results')
    if os.path.isdir('/content/extra_ckpts') and '/content/extra_ckpts' not in EXTRA_CKPT_DIRS: EXTRA_CKPT_DIRS.append('/content/extra_ckpts')
    print('EXTRA_RESULTS_DIRS =', EXTRA_RESULTS_DIRS)
except Exception as e:
    print('nothing uploaded (fine if all results are on this Drive):', e)

## 2. Load Results

In [ ]:
def _find(m, dirs):
    for d in dirs:
        p = os.path.join(d, f'{m}.json')
        if os.path.exists(p): return p
    return None
R = {}; _loaded = []; _missing = []
for m in METHODS:
    p = _find(m, [RESULTS_DIR] + EXTRA_RESULTS_DIRS)
    if p:
        R[m] = json.load(open(p)); _loaded.append(m); s = R[m]['summary']; md = R[m]['meta']
        where = '' if os.path.dirname(p) == RESULTS_DIR else f'  <- {os.path.dirname(p)}'
        print(f'loaded  {m:24} {md["total_steps"]:>9,} steps  {md["wall_clock_sec"]/3600:5.2f}h  best MSE {s["best_mse"]:.5f}  best R2 {s["best_r2"]:.3f}{where}')
    else:
        _missing.append(m); print(f'MISSING {m:24} (searched {[RESULTS_DIR] + EXTRA_RESULTS_DIRS})')
print(f'\n=== {len(_loaded)}/{len(METHODS)} loaded: {_loaded}  |  missing: {_missing} ===')

## 3. Configuration

In [ ]:
for m in R:
    md = R[m]['meta']; print('='*64); print(LABELS[m]); print(f'  P={md["P"]:,} seed={md["seed"]} samples={md["samples"]:,}'); print('  config:', json.dumps(md['config']))

## 4. Compare curves (MSE + R^2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for m in R:
    c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
    ax[0].plot(hrs, c['test_mse'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[1].plot(hrs, c['test_r2'],  'o-', color=COLORS[m], label=LABELS[m])
ax[0].set_xlabel('wall-clock hours'); ax[0].set_ylabel('test MSE'); ax[0].set_yscale('log'); ax[0].set_title('Test MSE vs time'); ax[0].legend()
ax[1].set_xlabel('wall-clock hours'); ax[1].set_ylabel('R^2'); ax[1].set_title('Variance explained (R^2) vs time'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Cost & Memory

In [ ]:
def fwd_equiv(m):
    st = R[m]['meta']['total_steps']
    if m.startswith('three_factor'): return st * 2 * R[m]['meta']['config'].get('M', 0)
    if m.startswith('backprop'): return st * 3
    return st * 2
print(f'{"method":30}{"steps":>10}{"fwd-equiv":>14}{"wall h":>8}{"peak MB":>9}')
print('-'*71)
for m in R:
    md = R[m]['meta']
    print(f'{LABELS[m]:30}{md["total_steps"]:>10,}{fwd_equiv(m):>14,}{md["wall_clock_sec"]/3600:>8.2f}{md.get("peak_mem_mb", float("nan")):>9.1f}')

## Peak training memory — backprop vs. forward-only (no activation tape)

Measures the **activation tape** backprop must retain for its backward pass — memory the forward-only three-factor rule never allocates (it only *evaluates* the loss). Self-contained and unexecuted: run it to print the numbers next to the accuracy results. Exact (`torch.autograd.graph.saved_tensors_hooks`), at this section's training batch size.

In [ ]:
# ============================================================================
# Peak training memory: backprop vs the forward-only three-factor rule.
# Backprop must retain an "activation tape" (one saved tensor per layer op) so
# the backward pass can run; the forward-only rule only EVALUATES the loss, so
# it stores none of it -> training memory stays at the inference footprint.
# Measured exactly via autograd's own saved-tensor hooks (not an estimate).
# Also writes peak_training_memory.csv into the exports/ folder so it rides
# along in the results zip next to summary.csv. Self-contained + unexecuted.
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as F
_MEM_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def _activation_tape_bytes(model, xshape, lossfn, make_target):
    model = model.to(_MEM_DEVICE)
    _pp = {p.untyped_storage().data_ptr() for p in model.parameters()}
    _seen, _isp = {}, {}
    def _pack(t):
        st = t.untyped_storage(); dp = st.data_ptr()
        _seen[dp] = st.nbytes(); _isp[dp] = (dp in _pp); return t
    def _unpack(t): return t
    x = torch.randn(*xshape, device=_MEM_DEVICE)
    with torch.autograd.graph.saved_tensors_hooks(_pack, _unpack):
        out = model(x); loss = lossfn(out, make_target(out)); loss.backward()
    return sum(nb for dp, nb in _seen.items() if not _isp[dp])  # activations only (params excluded)

def _report_training_memory(label, model, xshape, lossfn, make_target):
    P = sum(p.numel() for p in model.parameters()); pmem = P*4
    tape = _activation_tape_bytes(model, xshape, lossfn, make_target)
    common = 4*pmem; bp, loc = common + tape, common; MB = 1e6
    print("="*78)
    print(f"PEAK TRAINING MEMORY  |  {label}")
    print(f"  params P = {P:,}  ({pmem/MB:.2f} MB weights)   |   batch = {xshape[0]}   |   device = {_MEM_DEVICE}")
    print("-"*78)
    print(f"  activation tape backprop must retain : {tape/MB:8.2f} MB   ({tape/max(pmem,1):.1f}x the weights)")
    print(f"  forward-only three-factor rule       : {0.0:8.2f} MB   (no tape -- structural)")
    print(f"  est. training memory (same Adam both = 4P shared):")
    print(f"      backprop     : {bp/MB:8.2f} MB")
    print(f"      forward-only : {loc/MB:8.2f} MB   ->  backprop needs {bp/max(loc,1):.2f}x the memory")
    print(f"  (tape ~ linear in batch; measured via torch.autograd saved_tensors_hooks)")
    print("="*78)
    return {"section": label, "arch": type(model).__name__.replace("_Mem",""),
            "params": P, "batch": xshape[0], "weights_MB": round(pmem/MB, 2),
            "activation_tape_MB": round(tape/MB, 2), "tape_over_weights": round(tape/max(pmem,1), 2),
            "backprop_train_MB": round(bp/MB, 2), "forward_only_train_MB": round(loc/MB, 2),
            "backprop_over_forward_only": round(bp/max(loc,1), 2)}

class _MemMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__(); dims=[in_dim]+list(hidden)+[out_dim]
        self.layers=nn.ModuleList([nn.Linear(dims[i],dims[i+1]) for i in range(len(dims)-1)])
    def forward(self,x):
        for i,l in enumerate(self.layers):
            x=l(x)
            if i<len(self.layers)-1: x=torch.tanh(x)
        return x
_MEM_ROW = _report_training_memory("S6 MLP (student/teacher regression)",
    _MemMLP(64,(1024,896),16), (256,64),
    lambda o,t: F.mse_loss(o,t), lambda o: torch.randn_like(o))

# --- stash + persist so the export cell's zip picks it up (rides next to summary.csv) ---
PEAK_MEM_ROWS = [_MEM_ROW]
try:
    import os as _os, csv as _csv
    if 'STORE' in globals():
        _ed = _os.path.join(STORE, 'exports'); _os.makedirs(_ed, exist_ok=True)
        with open(_os.path.join(_ed, 'peak_training_memory.csv'), 'w', newline='') as _f:
            _w = _csv.DictWriter(_f, fieldnames=list(PEAK_MEM_ROWS[0].keys())); _w.writeheader()
            for _r in PEAK_MEM_ROWS: _w.writerow(_r)
        print('  saved ->', _os.path.join(_ed, 'peak_training_memory.csv'))
    else:
        print('  (STORE not defined yet -- CSV will be written by the export cell / re-run after Setup)')
except Exception as _e:
    print('  (peak_training_memory.csv not written:', _e, ')')


## 6. Head-to-head — three-factor cos sweep

In [ ]:
# three-factor cos sweep: clean vs normal vs noisy in equal wall-clock (does more-but-noisier keep winning?)
tf = [m for m in ['three_factor_clean', 'three_factor_normal', 'three_factor_noisy'] if m in R]
if len(tf) >= 2:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
    for m in tf:
        c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
        ax[0].plot(hrs, c['test_mse'], 'o-', color=COLORS[m], label=LABELS[m])
        ax[1].plot(hrs, c['test_r2'],  'o-', color=COLORS[m], label=LABELS[m])
        ax[2].plot(c['step'], c['test_mse'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[0].set_yscale('log'); ax[0].set_xlabel('hours'); ax[0].set_ylabel('test MSE'); ax[0].set_title('cos sweep — MSE vs time'); ax[0].legend()
    ax[1].set_xlabel('hours'); ax[1].set_ylabel('R^2'); ax[1].set_title('R^2 vs time'); ax[1].legend()
    ax[2].set_xscale('symlog'); ax[2].set_yscale('log'); ax[2].set_xlabel('steps'); ax[2].set_ylabel('test MSE'); ax[2].set_title('MSE vs steps'); ax[2].legend()
    plt.tight_layout(); plt.show()
    print(f'{"variant":30}{"M":>10}{"cos~":>8}{"steps":>10}{"best_mse":>12}{"best_r2":>10}')
    print('-'*80)
    for m in tf:
        md, s = R[m]['meta'], R[m]['summary']; M = md['config'].get('M') or 0; P = md['P']
        cos = (M/(M+P+1))**0.5 if M else float('nan')
        print(f'{LABELS[m]:30}{M:>10,}{cos:>8.3f}{md["total_steps"]:>10,}{s["best_mse"]:>12.5f}{s["best_r2"]:>10.3f}')
    best = min(tf, key=lambda m: R[m]['summary']['best_mse'])
    print(f'\nLowest best MSE in the budget: {LABELS[best]}. Reading down the cos column shows where fewer '
          f'probes / more steps stops paying off.')
else:
    print('Head-to-head needs >=2 of the three-factor variants (03/04/05).')

## 7. Predicted vs target (best model)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    """Deep tanh MLP. vmap-safe (Linear + tanh only)."""
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        dims = [in_dim] + list(hidden) + [out_dim]
        self.layers = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(dims)-1)])
    def forward(self, x):
        for i, lyr in enumerate(self.layers):
            x = lyr(x)
            if i < len(self.layers) - 1:
                x = torch.tanh(x)
        return x

# predicted vs target for the best (lowest-MSE) model — the regression analog of §8's predictions cell
import torch
def _ckpt(m):
    for d in [CKPT_DIR] + EXTRA_CKPT_DIRS:
        p = os.path.join(d, f'{m}.pt')
        if os.path.exists(p): return p
    return None
_avail = [m for m in R if _ckpt(m)]
if _avail:
    best = min(_avail, key=lambda m: R[m]['summary']['best_mse'])   # best model that has a checkpoint here
    cfg = R[best]['meta']['config']
    net = MLP(cfg['IN_DIM'], tuple(cfg['HIDDEN']), cfg['OUT_DIM']).to(device)
    ck = torch.load(_ckpt(best), map_location=device)['method']
    net.load_state_dict(ck['net'] if 'net' in ck else ck['params']); net.eval()
    torch.manual_seed(R[best]['meta']['seed'])
    teacher = torch.nn.Sequential(torch.nn.Linear(cfg['IN_DIM'],128), torch.nn.Tanh(), torch.nn.Linear(128,cfg['OUT_DIM'])).to(device)
    for p in teacher.parameters(): p.requires_grad_(False)
    x = torch.randn(1024, cfg['IN_DIM'], device=device)
    with torch.no_grad(): pred, tgt = net(x).cpu().flatten(), teacher(x).cpu().flatten()
    plt.figure(figsize=(5,5)); plt.scatter(tgt, pred, s=4, alpha=0.3, color=COLORS[best])
    lim = [min(tgt.min(),pred.min()), max(tgt.max(),pred.max())]; plt.plot(lim, lim, 'k--', lw=1)
    plt.xlabel('teacher target'); plt.ylabel('student prediction'); plt.title(f'Best model: {LABELS[best]} (fit quality)')
    plt.tight_layout(); plt.show()
else:
    print('scatter skipped: no checkpoints (.pt) on this Drive (upload one via the merge cell to see the fit).')

## 8. Summary table

In [ ]:
print(f'{"Experiment":30}{"Steps":>10}{"Init MSE":>11}{"Final MSE":>11}{"Best MSE":>11}{"Reduc %":>9}{"Best R2":>9}')
print('-'*91)
for m in R:
    md, s = R[m]['meta'], R[m]['summary']
    print(f'{LABELS[m]:30}{md["total_steps"]:>10,}{s["initial_mse"]:>11.5f}{s["final_mse"]:>11.5f}{s["best_mse"]:>11.5f}{s["reduction_pct"]:>9.1f}{s["best_r2"]:>9.3f}')

## 9. Export figures + CSVs to Drive (and download a zip)

In [ ]:
# Export everything: figures (PNG) + data (CSV) -> a Drive folder, and download a zip.
import os, csv, glob, shutil
EXPORT_DIR = os.path.join(STORE, 'exports'); os.makedirs(EXPORT_DIR, exist_ok=True)

# 1) CSVs -- long-format curves + a summary row per method
with open(os.path.join(EXPORT_DIR, 'curves.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['method', 't_sec', 'step', 'samples', 'train_mse', 'test_mse', 'test_r2'])
    for m in R:
        c = R[m]['curve']
        for i in range(len(c['t_sec'])):
            w.writerow([m, c['t_sec'][i], c['step'][i], c['samples'][i], c['train_mse'][i], c['test_mse'][i], c['test_r2'][i]])
with open(os.path.join(EXPORT_DIR, 'summary.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['method', 'P', 'total_steps', 'wall_clock_h', 'M', 'cos',
                                   'initial_mse', 'final_mse', 'best_mse', 'reduction_pct', 'final_r2', 'best_r2'])
    for m in R:
        md, s = R[m]['meta'], R[m]['summary']; M = md['config'].get('M') or 0
        cos = (M/(M+md['P']+1))**0.5 if M else ''
        w.writerow([m, md['P'], md['total_steps'], round(md['wall_clock_sec']/3600, 3), M, cos,
                    s['initial_mse'], s['final_mse'], s['best_mse'], round(s['reduction_pct'], 2), s['final_r2'], s['best_r2']])

# 2) Figures -- redraw the two comparison plots and save; also copy any per-run *_progress.png
def _save(fig, name): fig.savefig(os.path.join(EXPORT_DIR, name), dpi=120, bbox_inches='tight'); plt.close(fig)
if R:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    for m in R:
        c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
        ax[0].plot(hrs, c['test_mse'], 'o-', color=COLORS[m], label=LABELS[m]); ax[1].plot(hrs, c['test_r2'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[0].set_yscale('log'); ax[0].set_xlabel('hours'); ax[0].set_ylabel('test MSE'); ax[0].set_title('Test MSE vs time'); ax[0].legend()
    ax[1].set_xlabel('hours'); ax[1].set_ylabel('R^2'); ax[1].set_title('R^2 vs time'); ax[1].legend()
    _save(fig, 'curves_all_methods.png')
    tf = [m for m in ['three_factor_clean', 'three_factor_normal', 'three_factor_noisy'] if m in R]
    if len(tf) >= 2:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
        for m in tf:
            c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
            ax[0].plot(hrs, c['test_mse'], 'o-', color=COLORS[m], label=LABELS[m]); ax[1].plot(c['step'], c['test_mse'], 'o-', color=COLORS[m], label=LABELS[m])
        ax[0].set_yscale('log'); ax[0].set_xlabel('hours'); ax[0].set_title('cos sweep: MSE vs time'); ax[0].legend()
        ax[1].set_yscale('log'); ax[1].set_xscale('symlog'); ax[1].set_xlabel('steps'); ax[1].set_title('cos sweep: MSE vs steps'); ax[1].legend()
        _save(fig, 'cos_sweep.png')
for p in glob.glob(os.path.join(STORE, 'figures', '*_progress.png')):
    try: shutil.copy(p, EXPORT_DIR)
    except Exception: pass

print('exported to', EXPORT_DIR, '->', sorted(os.listdir(EXPORT_DIR)))

# 3) zip the folder + trigger a browser download (optional; the folder itself already lives on Drive)
_ztarget = '/content' if os.path.isdir('/content') else os.path.dirname(EXPORT_DIR)
_zip = shutil.make_archive(os.path.join(_ztarget, f'{DRIVE_SUBDIR}_exports'), 'zip', EXPORT_DIR)
try:
    from google.colab import files; files.download(_zip)
except Exception as e:
    print('download skipped (not in Colab):', e, '- zip at', _zip)